## 7.3 Cylindrical Maps
### 7.2.1 2D Datasets

In [ ]:
from pathlib import Path

from cartopy import crs as ccrs
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
from matplotlib import ticker
import matplotlib.pyplot as plt

import xarray as xr

import numpy as np
import pandas as pd
import numpy.ma as ma

In [ ]:
# Options to print figures into notebook/increase size
plt.rcParams.update({"font.size": 16, 
    "figure.figsize": [12, 6],
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]
    })

In [ ]:
filename = "hms_fire20250109.txt"
fires_path = Path("./data/txt") / filename
fires = pd.read_csv(fires_path, skipinitialspace=True)
fires.loc[fires["FRP"] < 0, "FRP"] = 0

In [ ]:
fires.head()

In [ ]:
carree_crs = ccrs.PlateCarree()

In [ ]:
fig = plt.figure(figsize=[10, 10])

ax = plt.subplot(projection=carree_crs)
ax.coastlines()
ax.set_global()

plt.scatter(fires["Lon"], fires["Lat"], s=1)
plt.show()

In [ ]:
lambert_crs = ccrs.LambertAzimuthalEqualArea(central_longitude=-100)

In [ ]:
fig = plt.figure(figsize=[10, 10])

ax = plt.subplot(projection=lambert_crs)

ax.coastlines()
ax.gridlines()
ax.set_global()

plt.scatter(fires["Lon"], fires["Lat"], s=1, transform=ccrs.PlateCarree())
plt.show()

In [ ]:
extent_los_angeles = [-120, -116.5, 32, 35]


In [ ]:
lon_labels = np.arange(-180, 180, 1.5)
lat_labels = np.arange(-90, 90, 1)

In [ ]:
fig = plt.figure()
ax = plt.subplot(projection=carree_crs)

ax.coastlines("50m")
ax.set_extent(extent_los_angeles)

frp = ax.scatter( \
    fires["Lon"], fires["Lat"], c=fires["FRP"], \
    s=fires["FRP"], transform=carree_crs, vmin=0, vmax=2000)
cbar = plt.colorbar(frp, orientation="horizontal", shrink=0.4)
cbar.set_label("Fire Radiative Power (MW)")

# 1) Maps the gridlines to the variable gl
gl = ax.gridlines(crs=carree_crs, draw_labels=True)

# 2) Adds two attributes to gl, which are xlocator and ylocator
gl.xlocator = ticker.FixedLocator(lon_labels)
gl.ylocator = ticker.FixedLocator(lat_labels)

# 3) Changes labels to show degrees North/South and East/West
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER

plt.show()

### 7.3.2 Swath Data

In [ ]:
filename = "JRR-AOD_v3r2_j01_s202501092048434_e202501092050080_c202501092128295.nc"
aod_file = Path("./data/aod") / filename
aod_nc = xr.open_dataset(aod_file)

In [ ]:
print(list(aod_nc.variables))

In [ ]:
aod = aod_nc["AOD550"]
lat = aod_nc["Latitude"]
lon = aod_nc["Longitude"]

In [ ]:
aod.shape, lat.shape, lon.shape

In [ ]:
aod.valid_range

In [ ]:
aod_levs = np.arange(0, 2.5, 0.1)
aod_levs

In [ ]:
fig = plt.figure(figsize=[12, 12])
ax = plt.subplot(projection=carree_crs)

ax.coastlines("50m")
ax.set_extent(extent_los_angeles)
x1 = ax.contourf(lon, lat, aod, aod_levs, extend="both", \
    transform=carree_crs)
fig.colorbar(x1, extend="both", orientation="horizontal", \
    fraction=0.05, pad=0.02)

ax.scatter(fires["Lon"], fires["Lat"], color="red", s=fires["FRP"]/10, alpha=0.1)

plt.show()

### 7.3.3 Quality Flag Filtering

In [ ]:
quality_flag = aod_nc["QCAll"]

mask_hq = (quality_flag != 0)
aod_hq = np.ma.masked_where(mask_hq, aod)

In [ ]:
percent_hq = (aod.count()-aod_hq.count())/aod.count()
percent_hq.item()

In [ ]:
fig, axis = plt.subplots(2, 1, subplot_kw={'projection': carree_crs})
upper_axis = axis[0]
lower_axis = axis[1]

axis[0].set_title("All Quality")
axis[0].coastlines("50m")
upper_fig = axis[0].contourf(lon, lat, aod, aod_levs, extend="both", transform=carree_crs)

axis[1].set_title("High Quality")
axis[1].coastlines("50m")
lower_fig = axis[1].contourf(lon, lat, aod_hq, aod_levs, extend="both", transform=carree_crs)

fig.colorbar(lower_fig, ax=axis, label="AOD")

plt.show()

# 7.3 Polar Stereographic Maps

In [ ]:
filename = "IS2SITMOGR4_01_202503_007_004.nc"
sea_ice_path = Path("./data/atlas") / filename
sea_ice = xr.open_dataset(sea_ice_path, engine="h5netcdf")
sea_ice

In [ ]:
sea_ice_2D = sea_ice["ice_thickness_int"][0,:,:]

In [ ]:
polar_crs = ccrs.NorthPolarStereo()

extent_arctic = [-180, 180, 60, 90]

plt.figure(figsize=[10, 10])
ax = plt.subplot(projection=polar_crs)
ax.coastlines("50m")
ax.gridlines()

ice_plot = ax.pcolormesh( \
    sea_ice["longitude"], sea_ice["latitude"], \
    sea_ice_2D, vmin=0, vmax=4, transform=carree_crs)
plt.colorbar(ice_plot, shrink=0.8, label="Ice Thickness (m)")

ax.set_extent(extent_arctic, crs=carree_crs)
plt.show()

# 7.4 Geostationary Maps

In [ ]:
filename = "OR_ABI-L1b-RadM2-M6C02_G19_s20252981300559_e20252981301017_c20252981301044.nc"
abi_path = Path("./data/abi") / filename
abi = xr.open_dataset(abi_path, engine="h5netcdf")
abi

In [ ]:
abi_rad = abi["Rad"]

In [ ]:
print("X:", X[0:3].values.round(0))
print("Y:", Y[0:3].values.round(0))

In [ ]:
proj_var = abi["goes_imager_projection"]
proj_var

In [ ]:
def get_sat_params(fid):
  geom = fid.goes_imager_projection

  # In meters and degrees
  satellite_params = {
  "central_lon": geom.longitude_of_projection_origin,
  "central_lat": 0.0,
  "sat_height": geom.perspective_point_height,
  "semi_major_axis" : geom.semi_major_axis,
  "semi_minor_axis" : geom.semi_minor_axis,
  "R_earth" : 6371000
  }

  return satellite_params

In [ ]:
abi_params = get_sat_params(abi)

In [ ]:
abi_params

In [ ]:
globe = ccrs.Globe(semimajor_axis=abi_params["semi_major_axis"], \
    semiminor_axis=abi_params["semi_minor_axis"])

In [ ]:
geo_crs = ccrs.Geostationary(central_longitude=abi_params["central_lon"], \
    satellite_height=abi_params["sat_height"], globe=globe)

In [ ]:
X = abi["x"] * abi_params["sat_height"]
Y = abi["y"] * abi_params["sat_height"]

abi_extent = (X.min(), X.max(), Y.min(), Y.max())

In [ ]:
fig = plt.figure(figsize=(10, 10))

ax = plt.subplot(projection=geo_crs)
ax.coastlines("10m", color="yellow", linewidth=2)

ax.imshow(abi_rad, origin="upper", cmap="gray", extent=abi_extent)
ax.set_extent(abi_extent, crs=geo_crs)

plt.show()

In [ ]:
filename = "OR_GLM-L2-LCFA_G19_s20252981300000_e20252981300200_c20252981300220.nc"
glm_path = Path("./data/glm") / filename
glm = xr.open_dataset(glm_path, engine="h5netcdf")
glm

In [ ]:
glm_lon = glm["event_lon"]
glm_lat = glm["event_lat"]
glm_area = glm["event_energy"]

glm_df = pd.DataFrame({"lat": glm_lat, "lon": glm_lon, \
    "area": glm_area })

In [ ]:
glm_area

In [ ]:
plt.figure(figsize=[10, 10])

ax = plt.subplot(projection=geo_crs)
ax.coastlines("10m", color="yellow", linewidth=2)

abi_im = ax.imshow(abi_rad, origin="upper", cmap="gray", extent=abi_extent)
glm_scatter = plt.scatter(glm_df["lon"], glm_df["lat"], c=glm_df["area"], \
    transform=carree_crs, s = 50, marker=".", vmin=0, vmax=3e-15)
ax.set_extent(abi_extent, crs=geo_crs)

plt.colorbar(glm_scatter, extend="both", shrink=0.75, label="Lightning Event Energy (J)")

plt.show()

# 7.5 Orthographic Maps

In [ ]:
# url = "http://psl.noaa.gov/thredds/dodsC/Datasets/noaa.ersst.v6/sst.mnmean.nc"
# ersst = xr.open_dataset(url)
# print(ersst)

filename = "sst.mnmean.nc"
glm_path = Path("./data/ersst") / filename
ersst = xr.open_dataset(glm_path, engine="h5netcdf")
ersst

In [ ]:
sst = ersst["sst"]

In [ ]:
most_recent = len(sst["time"].values)-1
sst_most_recent = sst.isel(time=most_recent)

In [ ]:
ortho_crs = ccrs.Orthographic(-90, 0)

fig = plt.figure(figsize=[10, 5])
ax = plt.subplot(projection=ortho_crs)
sst_most_recent.plot( \
    cmap=plt.get_cmap("plasma"), \
vmin=0, vmax=30, transform=carree_crs)
ax.coastlines("50m")
plt.show()

In [ ]:
levels = np.arange(0, 30, 2)

fig = plt.figure(figsize=[10, 5])
ax = plt.subplot(projection=ortho_crs)
sst_most_recent.plot.contourf( \
    levels=levels, \
    cmap=plt.get_cmap("plasma"), \
    transform=carree_crs)
ax.coastlines("50m")
plt.show()